# Smart Batching Search - Testing Notebook

This notebook demonstrates and tests the smart batching search functionality.

## Features
- **Planning**: Organize search using smart batching
- **Execution**: Execute search with proportional sampling
- **Rate Limiting**: Configurable requests per minute
- **Parallel Processing**: Efficient parallel execution

## Configuration

**Environment Variables:** This notebook loads configuration from a `.env` file in the `Smart_Batching` directory.

Create a `.env` file with:
```
BIGDATA_API_KEY=your_api_key_here
BIGDATA_API_BASE_URL=https://api.bigdata.com
```

**Options for API_BASE_URL:**
- `https://api.bigdata.com` (production - default)

**Note:** You must restart the kernel and run cells from the beginning if you change the API Base URL, as it's read at import time.

## 1. Load Environment Variables and Setup

**IMPORTANT:** Load `.env` file and set API_BASE_URL here before importing modules, as it's read at import time.

In [11]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from .env file
# Look for .env in the current directory (Smart_Batching folder)
env_path = Path.cwd() / ".env"
if env_path.exists():
    load_dotenv(env_path)
    print(f"✅ Loaded environment variables from {env_path}")
else:
    print(f"⚠️  .env file not found at {env_path}")
    print("   Using environment variables or defaults")

# Set API base URL BEFORE importing search_function
# This is important because smart_batching_config reads it at import time
API_BASE_URL = os.getenv("BIGDATA_API_BASE_URL", "https://api.bigdata.com")
os.environ["BIGDATA_API_BASE_URL"] = API_BASE_URL

print(f"✅ API Base URL configured: {API_BASE_URL}")

# Add current directory to path
sys.path.insert(0, str(Path.cwd()))

from search_function import (
    plan_search,
    execute_search,
    save_plan,
    load_plan,
    load_universe_from_csv
)

print("✅ Imports successful")

✅ Loaded environment variables from /Users/franciscogomez/git/bigdata/bigdata-cookbook/Smart_Batching/.env
✅ API Base URL configured: https://api.bigdata.com
✅ Imports successful


## 2. Configuration

In [12]:
# Configuration
# Note: API_BASE_URL and API_KEY are loaded from .env file in cell 2
# To change them, edit the .env file or set environment variables:
#   export BIGDATA_API_KEY='your_api_key_here'
#   export BIGDATA_API_BASE_URL='https://api.bigdata.com'

# API Key (loaded from .env in cell 2)
API_KEY = os.getenv("BIGDATA_API_KEY")

if not API_KEY:
    print("⚠️  BIGDATA_API_KEY not set. Please set it in your .env file:")
    print("   BIGDATA_API_KEY=your_api_key_here")
    print("   Or set environment variable: export BIGDATA_API_KEY='your_api_key_here'")
else:
    print(f"✅ API Key configured: {API_KEY[:8]}...{API_KEY[-4:]}")

# Test parameters
TEST_TEXT = "the company announced a new product"
TEST_UNIVERSE_CSV = "test_data/us_top3000.csv"  # Use small test universe
# TEST_UNIVERSE_CSV = "us_top3000.csv"  # Use full universe
TEST_START_DATE = "2021-01-01"
TEST_END_DATE = "2021-01-31"
TEST_CHUNK_PERCENTAGE = 0.1  # 10% of total chunks

print(f"\n📝 Test Configuration:")
print(f"   API Base URL: {API_BASE_URL}")
print(f"   Text: '{TEST_TEXT}'")
print(f"   Universe: {TEST_UNIVERSE_CSV}")
print(f"   Date Range: {TEST_START_DATE} to {TEST_END_DATE}")
print(f"   Chunk Percentage: {TEST_CHUNK_PERCENTAGE*100:.0f}%")

✅ API Key configured: bd_v2_pz...pj04

📝 Test Configuration:
   API Base URL: https://api.bigdata.com
   Text: 'the company announced a new product'
   Universe: test_data/us_top3000.csv
   Date Range: 2021-01-01 to 2021-01-31
   Chunk Percentage: 10%


## 3. Test Universe Loading

In [13]:
# Test loading universe from CSV
try:
    companies = load_universe_from_csv(TEST_UNIVERSE_CSV)
    print(f"✅ Loaded {len(companies)} companies from {TEST_UNIVERSE_CSV}")
    print(f"   First 5 companies: {companies[:5]}")
except Exception as e:
    print(f"❌ Error loading universe: {e}")

2026-01-22 16:27:08,398 - INFO - Loaded 4731 entity IDs from test_data/us_top3000.csv
✅ Loaded 4731 companies from test_data/us_top3000.csv
   First 5 companies: ['00067A', '001F1B', '002A99', '00326D', '003B70']


## 4. Step 1: Plan Search

In [4]:
# Plan the search
if API_KEY:
    print("📋 Planning search...")
    print("-" * 80)
    
    try:
        plan = plan_search(
            text=TEST_TEXT,
            universe_csv_path=TEST_UNIVERSE_CSV,
            start_date=TEST_START_DATE,
            end_date=TEST_END_DATE,
            api_key=API_KEY,
            api_base_url=API_BASE_URL
        )
        
        print(f"\n✅ Planning complete!")
        print(f"   Total expected chunks: {plan['total_expected_chunks']:,}")
        print(f"   Number of baskets: {len(plan['baskets'])}")
        
        if plan.get('planning_metadata'):
            metadata = plan['planning_metadata']
            print(f"   Total companies: {metadata.get('total_companies', 'N/A')}")
            print(f"   Companies with chunks: {metadata.get('companies_with_chunks', 'N/A')}")
            print(f"   Uses smart batching: {metadata.get('uses_smart_batching', False)}")
        
        # Show example basket
        if plan['baskets']:
            example_basket = plan['baskets'][0]
            print(f"\n   Example Basket:")
            print(f"     Basket ID: {example_basket['basket_id']}")
            print(f"     Expected chunks: {example_basket['expected_chunks']}")
            print(f"     Companies: {len(example_basket['companies'])} companies")
            print(f"     Query text: '{example_basket['query']['text']}'")
            print(f"     Max chunks in query: {example_basket['query']['max_chunks']}")
        
    except Exception as e:
        print(f"❌ Error during planning: {e}")
        import traceback
        traceback.print_exc()
        plan = None
else:
    print("⚠️  Skipping planning - API key not set")
    plan = None

📋 Planning search...
--------------------------------------------------------------------------------
2026-01-22 16:21:51,500 - INFO - Planning search for text: 'the company announced a new product'
2026-01-22 16:21:51,501 - INFO - Date range: 2021-01-01 to 2021-01-31
2026-01-22 16:21:51,502 - INFO - Loaded 4731 entity IDs from test_data/us_top3000.csv
2026-01-22 16:21:51,502 - INFO - Loaded 4731 companies from universe
2026-01-22 16:21:51,503 - INFO - Using SmartBatchingPlanner for optimized batching
    Querying 4731 companies in batches of 500 (estimated 10 queries)...
      Query 1/10: Found 234 companies from universe batch (out of 500 input)
      Query 2/10: Found 252 companies from universe batch (out of 500 input)
      Query 3/10: Found 243 companies from universe batch (out of 500 input)
      Query 4/10: Found 234 companies from universe batch (out of 500 input)
      Query 5/10: Found 249 companies from universe batch (out of 500 input)
      Query 6/10: Found 252 companie

## 5. Save Plan (Optional)

In [5]:
# Save plan for later use
if plan:
    plan_file = "test_search_plan.json"
    try:
        save_plan(plan, plan_file)
        print(f"✅ Plan saved to {plan_file}")
        print(f"   You can load it later with: plan = load_plan('{plan_file}')")
    except Exception as e:
        print(f"❌ Error saving plan: {e}")

2026-01-22 16:23:10,558 - INFO - Plan saved to test_search_plan.json
✅ Plan saved to test_search_plan.json
   You can load it later with: plan = load_plan('test_search_plan.json')


## 6. Step 2: Execute Search with Proportional Sampling

In [6]:
# Execute search with proportional sampling
if plan and API_KEY:
    print("🔍 Executing search...")
    print("-" * 80)
    
    try:
        results = execute_search(
            search_plan=plan,
            chunk_percentage=TEST_CHUNK_PERCENTAGE,
            requests_per_minute=100,  # Rate limit
            api_key=API_KEY,
            api_base_url=API_BASE_URL,
            sort_results=True,
            deduplicate_results=False
        )
        
        print(f"\n✅ Search complete!")
        print(f"   Retrieved {len(results):,} chunks")
        
        if results:
            print(f"\n   Top Results (by relevance):")
            for i, chunk in enumerate(results[:5], 1):
                print(f"\n   {i}. Relevance: {chunk.get('relevance', 0):.3f}")
                print(f"      Text: {chunk.get('text', '')[:100]}...")
                print(f"      Document: {chunk.get('document_id', 'N/A')}")
                print(f"      Source: {chunk.get('source_name', 'N/A')}")
                print(f"      Timestamp: {chunk.get('timestamp', 'N/A')}")
            
            # Save results as JSON (dataframe format)
            import json
            import pandas as pd
            from datetime import datetime
            
            # Convert results to DataFrame
            df = pd.DataFrame(results)
            
            # Save as JSON
            results_file = f"search_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
            df.to_json(results_file, orient='records', indent=2)
            print(f"\n💾 Results saved to {results_file}")
            print(f"   Total records: {len(df):,}")
            print(f"   Columns: {', '.join(df.columns.tolist()[:10])}...")
        else:
            print("   No results returned")
            results_file = None
            
    except Exception as e:
        print(f"❌ Error during execution: {e}")
        import traceback
        traceback.print_exc()
        results = []
else:
    print("⚠️  Skipping execution - plan or API key not available")
    results = []

🔍 Executing search...
--------------------------------------------------------------------------------
2026-01-22 16:23:10,569 - INFO - Executing search with 10.0% of chunks
2026-01-22 16:23:10,569 - INFO - Total expected chunks: 69,125
2026-01-22 16:23:10,570 - INFO - Searching 73 baskets
2026-01-22 16:23:11,919 - INFO - Basket medium_basket_2: Retrieved 61 chunks
2026-01-22 16:23:12,366 - INFO - Basket medium_basket_3: Retrieved 94 chunks
2026-01-22 16:23:12,416 - INFO - Basket medium_basket_5: Retrieved 68 chunks
2026-01-22 16:23:12,439 - INFO - Basket medium_basket_1: Retrieved 71 chunks
2026-01-22 16:23:12,505 - INFO - Basket medium_basket_6: Retrieved 86 chunks
2026-01-22 16:23:12,846 - INFO - Basket medium_basket_4: Retrieved 79 chunks
2026-01-22 16:23:13,494 - INFO - Basket medium_basket_0: Retrieved 73 chunks
2026-01-22 16:23:13,566 - INFO - Basket medium_basket_7: Retrieved 74 chunks
2026-01-22 16:23:17,127 - INFO - Basket low_basket_29: Retrieved 83 chunks
2026-01-22 16:23:1

## 7. Test Different Percentages

In [7]:
# Test with different chunk percentages
if plan and API_KEY:
    print("📊 Testing different chunk percentages...")
    print("-" * 80)
    
    percentages = [0.05, 0.1, 0.25, 0.5]
    
    for pct in percentages:
        try:
            results = execute_search(
                search_plan=plan,
                chunk_percentage=pct,
                requests_per_minute=100,
                api_key=API_KEY,
                api_base_url=API_BASE_URL,
                sort_results=True
            )
            print(f"   {pct*100:3.0f}%: {len(results):,} chunks retrieved")
        except Exception as e:
            print(f"   {pct*100:3.0f}%: Error - {e}")
else:
    print("⚠️  Skipping - plan or API key not available")

📊 Testing different chunk percentages...
--------------------------------------------------------------------------------
2026-01-22 16:23:57,658 - INFO - Executing search with 5.0% of chunks
2026-01-22 16:23:57,659 - INFO - Total expected chunks: 69,125
2026-01-22 16:23:57,659 - INFO - Searching 73 baskets
2026-01-22 16:23:58,950 - INFO - Basket medium_basket_5: Retrieved 37 chunks
2026-01-22 16:23:59,456 - INFO - Basket medium_basket_2: Retrieved 31 chunks
2026-01-22 16:23:59,473 - INFO - Basket medium_basket_7: Retrieved 37 chunks
2026-01-22 16:23:59,481 - INFO - Basket medium_basket_3: Retrieved 46 chunks
2026-01-22 16:23:59,492 - INFO - Basket medium_basket_1: Retrieved 36 chunks
2026-01-22 16:23:59,538 - INFO - Basket medium_basket_0: Retrieved 34 chunks
2026-01-22 16:23:59,586 - INFO - Basket medium_basket_6: Retrieved 43 chunks
2026-01-22 16:23:59,626 - INFO - Basket medium_basket_4: Retrieved 37 chunks
2026-01-22 16:24:03,983 - INFO - Basket medium_basket_8: Retrieved 36 chunk

## 8. Analyze Results

In [8]:
# Analyze results if available
if results:
    print("📈 Results Analysis")
    print("-" * 80)
    
    # Relevance distribution
    relevances = [chunk.get('relevance', 0) for chunk in results]
    if relevances:
        print(f"\n   Relevance Scores:")
        print(f"     Min: {min(relevances):.3f}")
        print(f"     Max: {max(relevances):.3f}")
        print(f"     Avg: {sum(relevances)/len(relevances):.3f}")
    
    # Sentiment distribution
    sentiments = [chunk.get('sentiment', 0) for chunk in results if chunk.get('sentiment') is not None]
    if sentiments:
        positive = sum(1 for s in sentiments if s > 0)
        negative = sum(1 for s in sentiments if s < 0)
        neutral = len(sentiments) - positive - negative
        print(f"\n   Sentiment Distribution:")
        print(f"     Positive: {positive} ({positive/len(sentiments)*100:.1f}%)")
        print(f"     Negative: {negative} ({negative/len(sentiments)*100:.1f}%)")
        print(f"     Neutral: {neutral} ({neutral/len(sentiments)*100:.1f}%)")
    
    # Source distribution
    sources = {}
    for chunk in results:
        source = chunk.get('source_name', 'Unknown')
        sources[source] = sources.get(source, 0) + 1
    
    if sources:
        print(f"\n   Top Sources:")
        sorted_sources = sorted(sources.items(), key=lambda x: x[1], reverse=True)
        for source, count in sorted_sources[:5]:
            print(f"     {source}: {count} chunks")
    

📈 Results Analysis
--------------------------------------------------------------------------------

   Relevance Scores:
     Min: 0.030
     Max: 0.428
     Avg: 0.087

   Sentiment Distribution:
     Positive: 24998 (86.4%)
     Negative: 3099 (10.7%)
     Neutral: 839 (2.9%)

   Top Sources:
     Benzinga: 14939 chunks
     Factset Transcripts: 3990 chunks
     Quartr Reports: 2379 chunks
     The Fly: 1182 chunks
     Quartr Transcripts: 1077 chunks


## 9. Load Saved Plan (Optional)

In [9]:
# Load a previously saved plan
plan_file = "test_search_plan.json"

if os.path.exists(plan_file):
    try:
        loaded_plan = load_plan(plan_file)
        print(f"✅ Plan loaded from {plan_file}")
        print(f"   Total expected chunks: {loaded_plan.get('total_expected_chunks', 0):,}")
        print(f"   Number of baskets: {len(loaded_plan.get('baskets', []))}")
        print(f"\n   You can now execute with different percentages:")
        print(f"   results = execute_search(loaded_plan, chunk_percentage=0.2)")
    except Exception as e:
        print(f"❌ Error loading plan: {e}")
else:
    print(f"ℹ️  Plan file '{plan_file}' not found. Save a plan first.")

2026-01-22 16:27:08,355 - INFO - Plan loaded from test_search_plan.json
✅ Plan loaded from test_search_plan.json
   Total expected chunks: 69,125
   Number of baskets: 73

   You can now execute with different percentages:
   results = execute_search(loaded_plan, chunk_percentage=0.2)


## 10. Summary

In [10]:
print("=" * 80)
print("Smart Batching Search - Test Summary")
print("=" * 80)

if plan:
    print(f"✅ Planning: SUCCESS")
    print(f"   Expected chunks: {plan['total_expected_chunks']:,}")
    print(f"   Baskets created: {len(plan['baskets'])}")
else:
    print("⚠️  Planning: Not completed")

if results:
    print(f"✅ Execution: SUCCESS")
    print(f"   Chunks retrieved: {len(results):,}")
    print(f"   Percentage used: {TEST_CHUNK_PERCENTAGE*100:.0f}%")
    if plan:
        expected = plan['total_expected_chunks']
        actual = len(results)
        if expected > 0:
            print(f"   Actual vs Expected: {actual/expected*100:.1f}%")
else:
    print("⚠️  Execution: Not completed")

print("\n" + "=" * 80)
print("Test complete!")
print("=" * 80)

Smart Batching Search - Test Summary
✅ Planning: SUCCESS
   Expected chunks: 69,125
   Baskets created: 73
✅ Execution: SUCCESS
   Chunks retrieved: 28,936
   Percentage used: 10%
   Actual vs Expected: 41.9%

Test complete!
